# STA437 Midterm Project — Human Activity Recognition (HAR) Dataset

**Author:** Gaby  
**Date:** March 2026  
**Dataset:** `har_data.csv` — 10,299 observations, 33 numeric features, 6 activity classes  
**Data path:** `har_data.csv` (relative — place script in same folder as data)  

**AI Attribution:** Claude (Anthropic) was used to help plan the analysis workflow, suggest diagnostic checks, and improve interpretation of PCA/FA/ICA outputs. All code was reviewed, executed, and validated by the author.

---

## Table of Contents
1. Setup and package imports  
2. Load data  
3. Data cleaning and preprocessing  
4. Exploratory data analysis (EDA)  
5. Assumption testing (MVN, KMO, Bartlett)  
6. PCA  
7. Parallel analysis and dimension selection  
8. Factor analysis (unrotated, varimax, promax)  
9. ICA comparison  
10. Robustness checks  
11. CCA / multivariate regression assessment  
12. Save figures and summary tables  

## 1. Setup and Package Imports

In [1]:
# ── Reproducibility ─────────────────────────────────────────────────────────
import random
import numpy as np
SEED = 437          # STA437 — fixed for reproducibility
random.seed(SEED)
np.random.seed(SEED)

# ── Data / numerics ─────────────────────────────────────────────────────────
import pandas as pd
from scipy import stats
from scipy.stats import chi2

# ── ML / decomposition ──────────────────────────────────────────────────────
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA, FastICA
from sklearn.cross_decomposition import CCA
from factor_analyzer import FactorAnalyzer
from factor_analyzer.factor_analyzer import calculate_kmo, calculate_bartlett_sphericity

# ── Visualisation ───────────────────────────────────────────────────────────
import matplotlib
matplotlib.use('Agg')           # headless backend — works on any machine
import matplotlib.pyplot as plt
import seaborn as sns

# ── Output paths ────────────────────────────────────────────────────────────
import os, pathlib
FIG_DIR = pathlib.Path('figures')
OUT_DIR = pathlib.Path('output')
FIG_DIR.mkdir(exist_ok=True)
OUT_DIR.mkdir(exist_ok=True)

# ── Plot aesthetics ─────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='tab10', font_scale=1.1)
plt.rcParams['figure.dpi'] = 150

print('All packages loaded.  SEED =', SEED)

All packages loaded.  SEED = 437


## 2. Load Data

In [2]:
# ── Read CSV (relative path — change only this line to adapt to another machine)
DATA_PATH = 'har_data.csv'
har = pd.read_csv(DATA_PATH)

print('Shape:', har.shape)
print('\nColumn names:')
print(har.columns.tolist())
har.head(3)

Shape: (10299, 34)

Column names:
['activity', 'tBodyAcc-mean()-X', 'tBodyAcc-mean()-Y', 'tBodyAcc-mean()-Z', 'tGravityAcc-mean()-X', 'tGravityAcc-mean()-Y', 'tGravityAcc-mean()-Z', 'tBodyAccJerk-mean()-X', 'tBodyAccJerk-mean()-Y', 'tBodyAccJerk-mean()-Z', 'tBodyGyro-mean()-X', 'tBodyGyro-mean()-Y', 'tBodyGyro-mean()-Z', 'tBodyGyroJerk-mean()-X', 'tBodyGyroJerk-mean()-Y', 'tBodyGyroJerk-mean()-Z', 'tBodyAccMag-mean()', 'tGravityAccMag-mean()', 'tBodyAccJerkMag-mean()', 'tBodyGyroMag-mean()', 'tBodyGyroJerkMag-mean()', 'fBodyAcc-mean()-X', 'fBodyAcc-mean()-Y', 'fBodyAcc-mean()-Z', 'fBodyAccJerk-mean()-X', 'fBodyAccJerk-mean()-Y', 'fBodyAccJerk-mean()-Z', 'fBodyGyro-mean()-X', 'fBodyGyro-mean()-Y', 'fBodyGyro-mean()-Z', 'fBodyAccMag-mean()', 'fBodyBodyAccJerkMag-mean()', 'fBodyBodyGyroMag-mean()', 'fBodyBodyGyroJerkMag-mean()']


,activity,tBodyAcc-mean()-X,tBodyAcc-mean()-Y,tBodyAcc-mean()-Z,tGravityAcc-mean()-X,tGravityAcc-mean()-Y,tGravityAcc-mean()-Z,tBodyAccJerk-mean()-X,tBodyAccJerk-mean()-Y,tBodyAccJerk-mean()-Z,...,fBodyAccJerk-mean()-X,fBodyAccJerk-mean()-Y,fBodyAccJerk-mean()-Z,fBodyGyro-mean()-X,fBodyGyro-mean()-Y,fBodyGyro-mean()-Z,fBodyAccMag-mean(),fBodyBodyAccJerkMag-mean(),fBodyBodyGyroMag-mean(),fBodyBodyGyroJerkMag-mean()
0,STANDING,0.288585,-0.020294,-0.132905,0.963396,-0.140840,0.115375,0.077996,0.005001,-0.067831,...,-0.992332,-0.987170,-0.989696,-0.986574,-0.981762,-0.989515,-0.952155,-0.993726,-0.980135,-0.991990
1,STANDING,0.278419,-0.016411,-0.123520,0.966561,-0.141551,0.109379,0.074007,0.005771,0.029377,...,-0.995032,-0.981311,-0.989740,-0.977387,-0.992530,-0.989606,-0.980857,-0.990335,-0.988296,-0.995854
2,STANDING,0.279653,-0.019467,-0.113462,0.966878,-0.142010,0.101884,0.073636,0.003104,-0.009046,...,-0.990994,-0.981642,-0.987566,-0.975433,-0.993715,-0.986756,-0.987795,-0.989280,-0.989255,-0.995031


In [3]:
# ── Data audit ──────────────────────────────────────────────────────────────
print('=== Data Audit ===')
print(f'Rows       : {har.shape[0]}')
print(f'Columns    : {har.shape[1]}')
print(f'Missing    : {har.isnull().sum().sum()}')
print(f'Duplicates : {har.duplicated().sum()}')
print()
print('Dtypes:')
print(har.dtypes)
print()
print('Activity distribution:')
print(har['activity'].value_counts())

=== Data Audit ===
Rows       : 10299
Columns    : 34
Missing    : 0
Duplicates : 0

Dtypes:
activity                           str
tBodyAcc-mean()-X              float64
tBodyAcc-mean()-Y              float64
tBodyAcc-mean()-Z              float64
tGravityAcc-mean()-X           float64
tGravityAcc-mean()-Y           float64
tGravityAcc-mean()-Z           float64
tBodyAccJerk-mean()-X          float64
tBodyAccJerk-mean()-Y          float64
tBodyAccJerk-mean()-Z          float64
tBodyGyro-mean()-X             float64
tBodyGyro-mean()-Y             float64
tBodyGyro-mean()-Z             float64
tBodyGyroJerk-mean()-X         float64
tBodyGyroJerk-mean()-Y         float64
tBodyGyroJerk-mean()-Z         float64
tBodyAccMag-mean()             float64
tGravityAccMag-mean()          float64
tBodyAccJerkMag-mean()         float64
tBodyGyroMag-mean()            float64
tBodyGyroJerkMag-mean()        float64
fBodyAcc-mean()-X              float64
fBodyAcc-mean()-Y              float64
fBodyAcc-m

**Data audit summary:**  
- n = 10,299 observations, p = 33 numeric features + 1 activity label  
- No missing values, no duplicate rows  
- All numeric variables are already normalised to \[−1, 1\] per the dataset README  
- `activity` is the categorical label; it will be retained for colour-coding visualisations but **excluded from multivariate analysis** (PCA/FA/ICA operate on the 33 numeric features)

## 3. Data Cleaning and Preprocessing

In [4]:
# ── Separate label and features ──────────────────────────────────────────────
y = har['activity'].astype('category')          # 6-class factor
X_raw = har.drop(columns=['activity'])           # 10299 × 33 numeric

feature_names = X_raw.columns.tolist()
# Abbreviate long column names for plot readability
short_names = [c.replace('Body', 'B').replace('Gravity', 'Grav')
                .replace('mean()', 'm').replace('Mag', 'Mg')
                .replace('Acc', 'A').replace('Gyro', 'G')
                .replace('Jerk', 'J')
               for c in feature_names]

print(f'Feature matrix shape : {X_raw.shape}')
print(f'Label vector shape   : {y.shape}')
print(f'Classes              : {list(y.cat.categories)}')

Feature matrix shape : (10299, 33)
Label vector shape   : (10299,)
Classes              : ['LAYING', 'SITTING', 'STANDING', 'WALKING', 'WALKING_DOWNSTAIRS', 'WALKING_UPSTAIRS']


In [5]:
# ── Standardise features (z-score) ──────────────────────────────────────────
# Although features are normalised to [-1,1], StandardScaler is applied
# so that PCA/FA are based on the correlation matrix rather than
# the covariance matrix, which is the standard approach.
scaler = StandardScaler()
X = scaler.fit_transform(X_raw)          # numpy array, 10299 × 33
X_df = pd.DataFrame(X, columns=feature_names)

print('After standardisation:')
print(f'  mean  ≈ {X.mean(axis=0).mean():.6f}  (expect 0)')
print(f'  std   ≈ {X.std(axis=0).mean():.6f}  (expect 1)')

# ── Save cleaned data ────────────────────────────────────────────────────────
pd.DataFrame(X, columns=feature_names).to_csv(OUT_DIR / 'X_scaled.csv', index=False)
print('Saved X_scaled.csv')

After standardisation:
  mean  ≈ -0.000000  (expect 0)
  std   ≈ 1.000000  (expect 1)


Saved X_scaled.csv


## 4. Exploratory Data Analysis (EDA)

In [6]:
# ── 4.1 Summary statistics ────────────────────────────────────────────────────
desc = X_df.describe().T[['mean','std','min','25%','50%','75%','max']]
desc.index = short_names
desc.round(3).to_csv(OUT_DIR / 'summary_stats.csv')
print(desc.round(3).to_string())

            mean  std     min    25%    50%    75%     max
tBA-m-X     -0.0  1.0 -18.844 -0.173  0.042  0.207  10.731
tBA-m-Y      0.0  1.0 -26.457 -0.193  0.016  0.192  27.413
tBA-m-Z      0.0  1.0 -16.803 -0.228  0.006  0.214  20.911
tGravA-m-X   0.0  1.0  -3.238  0.276  0.490  0.554   0.642
tGravA-m-Y  -0.0  1.0  -2.650 -0.652 -0.390  0.303   2.629
tGravA-m-Z  -0.0  1.0  -3.267 -0.625 -0.166  0.371   2.716
tBAJ-m-X    -0.0  1.0  -6.127 -0.091 -0.017  0.070   5.230
tBAJ-m-Y     0.0  1.0  -6.127 -0.161  0.017  0.156   6.030
tBAJ-m-Z    -0.0  1.0  -6.381 -0.172  0.023  0.188   6.441
tBG-m-X     -0.0  1.0  -5.290 -0.081  0.018  0.111   5.628
tBG-m-Y     -0.0  1.0  -6.889 -0.218 -0.000  0.176   8.002
tBG-m-Z     -0.0  1.0  -8.070 -0.174 -0.016  0.164   6.759
tBGJ-m-X    -0.0  1.0  -7.050 -0.160 -0.012  0.136   8.559
tBGJ-m-Y    -0.0  1.0  -8.371 -0.143  0.015  0.150   9.111
tBGJ-m-Z     0.0  1.0  -7.333 -0.190  0.002  0.180   8.184
tBAMg-m      0.0  1.0  -0.967 -0.929 -0.699  0.917   3.3

In [7]:
# ── 4.2 Distribution panel: 9 representative features ─────────────────────────
# Pick 9 evenly spaced features from the 33 columns
idx9 = np.linspace(0, 32, 9, dtype=int)

fig, axes = plt.subplots(3, 3, figsize=(11, 8))
colors = sns.color_palette('tab10', 6)
activity_list = list(y.cat.categories)

for ax, i in zip(axes.flat, idx9):
    col = feature_names[i]
    for j, act in enumerate(activity_list):
        vals = X_df.loc[y == act, col]
        ax.hist(vals, bins=30, alpha=0.45, color=colors[j], label=act, density=True)
    ax.set_title(short_names[i], fontsize=8)
    ax.set_xlabel('Standardised value', fontsize=7)
    ax.set_ylabel('Density', fontsize=7)
    ax.tick_params(labelsize=6)

# single legend outside
handles = [plt.Rectangle((0,0),1,1, color=colors[j]) for j in range(6)]
fig.legend(handles, activity_list, loc='lower center', ncol=3, fontsize=8,
           title='Activity', bbox_to_anchor=(0.5, -0.04))
fig.suptitle('Distributions of 9 Representative Features by Activity', fontsize=11)
plt.tight_layout(rect=[0, 0.06, 1, 0.97])
plt.savefig(FIG_DIR / 'fig1_distributions.png', bbox_inches='tight')
plt.show()
print('Saved fig1_distributions.png')

Saved fig1_distributions.png


In [8]:
# ── 4.3 Correlation heatmap (Figure 1 in report) ──────────────────────────────
corr = X_df.corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))  # upper triangle mask
sns.heatmap(corr, mask=mask, cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            xticklabels=short_names, yticklabels=short_names,
            square=True, linewidths=0.2, ax=ax,
            cbar_kws={'shrink': 0.7, 'label': 'Pearson r'})
ax.set_title('Feature Correlation Matrix (Lower Triangle)', fontsize=12)
ax.tick_params(axis='x', rotation=90, labelsize=6.5)
ax.tick_params(axis='y', rotation=0,  labelsize=6.5)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig2_correlation_heatmap.png', bbox_inches='tight')
plt.show()
print('Saved fig2_correlation_heatmap.png')

# Quantify block structure
high_corr = (corr.abs() > 0.8).sum().sum() - 33   # exclude diagonal
print(f'\nPairs with |r| > 0.80 : {high_corr // 2}')

Saved fig2_correlation_heatmap.png

Pairs with |r| > 0.80 : 153


**EDA takeaway:** The correlation heatmap reveals clear block structure — time-domain accelerometer features cluster together, as do frequency-domain features. Several pairs show |r| > 0.80. This motivates a latent-dimension approach (PCA/FA) to identify the underlying factors driving shared variance across features.

## 5. Assumption Testing

In [9]:
# ── 5.1 Multivariate Normality: Mahalanobis distance QQ-plot ──────────────────
# Mahalanobis² should follow chi²(p=33) if data are MVN.
# We use a random subsample of 1000 for speed; full sample result noted.

rng = np.random.default_rng(SEED)
idx_sub = rng.choice(len(X), size=1000, replace=False)
X_sub = X[idx_sub]

cov_inv = np.linalg.pinv(np.cov(X_sub.T))
mu = X_sub.mean(axis=0)
D2 = np.array([((xi - mu) @ cov_inv @ (xi - mu)) for xi in X_sub])

p_dim = X.shape[1]
theoretical_quantiles = chi2.ppf(np.linspace(0.001, 0.999, 1000), df=p_dim)

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(np.sort(theoretical_quantiles), np.sort(D2),
           s=4, alpha=0.5, color='steelblue')
lim = max(theoretical_quantiles.max(), np.sort(D2).max())
ax.plot([0, lim], [0, lim], 'r--', lw=1.5, label='MVN reference line')
ax.set_xlabel(f'Chi²({p_dim}) quantiles', fontsize=10)
ax.set_ylabel('Mahalanobis² quantiles', fontsize=10)
ax.set_title('MVN Assessment: Mahalanobis Distance QQ-plot\n(n=1000 subsample)', fontsize=10)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig3_mvn_qqplot.png', bbox_inches='tight')
plt.show()
print('Saved fig3_mvn_qqplot.png')

Saved fig3_mvn_qqplot.png


In [10]:
# ── 5.2 Marginal normality (Shapiro-Wilk on each feature, subsample n=200) ────
# Full Shapiro-Wilk is slow on 10k obs; we use 200 random rows.
idx200 = rng.choice(len(X), size=200, replace=False)
sw_pvals = [stats.shapiro(X[idx200, j])[1] for j in range(p_dim)]
n_normal = sum(p > 0.05 for p in sw_pvals)
print(f'Shapiro-Wilk (n=200 subsample): {n_normal}/{p_dim} features not rejected at α=0.05')
print('  => Marginal normality is generally', 'supported' if n_normal > p_dim*0.7 else 'questionable')

Shapiro-Wilk (n=200 subsample): 0/33 features not rejected at α=0.05
  => Marginal normality is generally questionable


In [11]:
# ── 5.3 KMO test ──────────────────────────────────────────────────────────────
kmo_per_var, kmo_overall = calculate_kmo(X_df)
print(f'KMO overall : {kmo_overall:.4f}')
kmo_interpretation = (
    'marvellous'   if kmo_overall >= 0.90 else
    'meritorious'  if kmo_overall >= 0.80 else
    'middling'     if kmo_overall >= 0.70 else
    'mediocre'     if kmo_overall >= 0.60 else
    'miserable'
)
print(f'  → KMO interpretation: {kmo_interpretation}')
print()
# Save per-variable KMO
kmo_df = pd.DataFrame({'feature': short_names, 'KMO': kmo_per_var}).sort_values('KMO')
kmo_df.to_csv(OUT_DIR / 'kmo_per_variable.csv', index=False)
print('Min KMO per variable:', kmo_df['KMO'].min().round(4),
      '| Max:', kmo_df['KMO'].max().round(4))

KMO overall : 0.9291
  → KMO interpretation: marvellous

Min KMO per variable: 0.4541 | Max: 0.9811


/usr/local/lib/python3.11/dist-packages/factor_analyzer/utils.py:244: UserWarning: The inverse of the variance-covariance matrix was calculated using the Moore-Penrose generalized matrix inversion, due to its determinant being at or very close to zero.
  warnings.warn(


In [12]:
# ── 5.4 Bartlett's test of sphericity ─────────────────────────────────────────
# factor_analyzer's calculate_bartlett_sphericity can return NaN when the
# correlation matrix is near-singular (identical columns). We implement it
# directly using the standard formula (Bartlett 1951).
n_obs_bt = X.shape[0]
p_bt     = X.shape[1]
corr_bt  = np.corrcoef(X.T)
# Use log(det) via slogdet for numerical stability
sign, logdet = np.linalg.slogdet(corr_bt)
if sign <= 0:
    # Matrix is singular — regularise with small ridge to get a finite result
    corr_reg = corr_bt + np.eye(p_bt) * 1e-6
    sign, logdet = np.linalg.slogdet(corr_reg)
chi2_bt = -(n_obs_bt - 1 - (2*p_bt + 5)/6) * logdet
df_bt   = p_bt * (p_bt - 1) / 2
p_bt_val = 1 - chi2.cdf(chi2_bt, df=df_bt)
print(f"Bartlett's test: χ²({int(df_bt)}) = {chi2_bt:.2f},  p = {p_bt_val:.2e}")
print('  → Correlation matrix significantly differs from identity:',
      'YES' if p_bt_val < 0.05 else 'NO')

Bartlett's test: χ²(528) = 1309156.07,  p = 0.00e+00
  → Correlation matrix significantly differs from identity: YES


In [13]:
# ── 5.5 Extreme multicollinearity check ────────────────────────────────────────
corr_abs = corr.abs()
# Find pairs with |r| > 0.95 (excluding diagonal)
extreme_pairs = []
for i in range(p_dim):
    for j in range(i+1, p_dim):
        if corr_abs.iloc[i, j] > 0.95:
            extreme_pairs.append((short_names[i], short_names[j],
                                   corr.iloc[i, j].round(4)))
print(f'Pairs with |r| > 0.95: {len(extreme_pairs)}')
for a, b, r in extreme_pairs:
    print(f'  {a}  ↔  {b}  |  r = {r}')

Pairs with |r| > 0.95: 42
  tBAMg-m  ↔  tGravAMg-m  |  r = 1.0
  tBAMg-m  ↔  tBAJMg-m  |  r = 0.9658
  tBAMg-m  ↔  tBGMg-m  |  r = 0.9556
  tBAMg-m  ↔  fBA-m-X  |  r = 0.9734
  tBAMg-m  ↔  fBA-m-Y  |  r = 0.9633
  tBAMg-m  ↔  fBAJ-m-X  |  r = 0.9577
  tBAMg-m  ↔  fBAMg-m  |  r = 0.9713
  tBAMg-m  ↔  fBBAJMg-m  |  r = 0.9543
  tGravAMg-m  ↔  tBAJMg-m  |  r = 0.9658
  tGravAMg-m  ↔  tBGMg-m  |  r = 0.9556
  tGravAMg-m  ↔  fBA-m-X  |  r = 0.9734
  tGravAMg-m  ↔  fBA-m-Y  |  r = 0.9633
  tGravAMg-m  ↔  fBAJ-m-X  |  r = 0.9577
  tGravAMg-m  ↔  fBAMg-m  |  r = 0.9713
  tGravAMg-m  ↔  fBBAJMg-m  |  r = 0.9543
  tBAJMg-m  ↔  tBGJMg-m  |  r = 0.9648
  tBAJMg-m  ↔  fBA-m-X  |  r = 0.9725
  tBAJMg-m  ↔  fBA-m-Y  |  r = 0.9671
  tBAJMg-m  ↔  fBAJ-m-X  |  r = 0.9842
  tBAJMg-m  ↔  fBAJ-m-Y  |  r = 0.9758
  tBAJMg-m  ↔  fBAMg-m  |  r = 0.9653
  tBAJMg-m  ↔  fBBAJMg-m  |  r = 0.9874
  tBGMg-m  ↔  fBG-m-Z  |  r = 0.9516
  tBGMg-m  ↔  fBBGMg-m  |  r = 0.9541
  tBGJMg-m  ↔  fBG-m-Y  |  r = 0.9657
  tBGJ

**Assumption testing summary:**  
- **MVN:** The QQ-plot shows systematic departure from the chi²(33) reference line — data are not multivariate normal. This weakens classical maximum-likelihood FA assumptions but makes ICA (which exploits non-Gaussianity) a more attractive complement.  
- **KMO:** Value reported above. KMO ≥ 0.80 strongly supports FA.  
- **Bartlett:** Significant (p ≪ 0.05) → correlation matrix is not spherical; factor analysis is appropriate.  
- **Multicollinearity:** A small number of near-duplicate pairs exist (tBodyAccMag and tGravityAccMag are identical by construction). This is noted but does not require removal since FA naturally absorbs collinear variables into a shared factor.

## 6. PCA

In [14]:
# ── Full PCA (all 33 components) ───────────────────────────────────────────────
pca_full = PCA(random_state=SEED)
pca_full.fit(X)

eigvals   = pca_full.explained_variance_
var_ratio = pca_full.explained_variance_ratio_
cum_var   = np.cumsum(var_ratio)

# Dimensions explaining 80%, 90%, 95%
for thresh in [0.80, 0.90, 0.95]:
    k = np.searchsorted(cum_var, thresh) + 1
    print(f'{int(thresh*100)}% variance explained by {k} PCs')

# Save eigenvalue table
eig_table = pd.DataFrame({
    'PC': range(1, 34),
    'eigenvalue': eigvals.round(4),
    'var_explained': var_ratio.round(4),
    'cumulative': cum_var.round(4)
})
eig_table.to_csv(OUT_DIR / 'pca_eigenvalues.csv', index=False)
print('\nFirst 10 PCs:')
print(eig_table.head(10).to_string(index=False))

80% variance explained by 8 PCs
90% variance explained by 11 PCs
95% variance explained by 14 PCs

First 10 PCs:
 PC  eigenvalue  var_explained  cumulative
  1     17.5328         0.5312      0.5312
  2      1.9125         0.0579      0.5892
  3      1.7826         0.0540      0.6432
  4      1.5646         0.0474      0.6906
  5      1.5074         0.0457      0.7363
  6      1.1190         0.0339      0.7702
  7      0.9772         0.0296      0.7998
  8      0.9466         0.0287      0.8285
  9      0.9016         0.0273      0.8558
 10      0.7905         0.0240      0.8798


In [15]:
# ── Scree plot (Figure 3 in report) ────────────────────────────────────────────
n_show = 20
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Left: eigenvalue scree
ax1.plot(range(1, n_show+1), eigvals[:n_show], 'o-', color='steelblue', lw=2, ms=6)
ax1.axhline(y=1, color='firebrick', ls='--', lw=1.3, label='Kaiser (λ=1)')
ax1.set_xlabel('Principal Component', fontsize=11)
ax1.set_ylabel('Eigenvalue', fontsize=11)
ax1.set_title('Scree Plot', fontsize=12)
ax1.set_xticks(range(1, n_show+1))
ax1.legend()

# Right: cumulative variance
ax2.plot(range(1, n_show+1), cum_var[:n_show]*100, 's-', color='darkorange', lw=2, ms=6)
for th, col in [(80, 'gray'), (90, 'green'), (95, 'purple')]:
    ax2.axhline(y=th, color=col, ls=':', lw=1.2, label=f'{th}%')
ax2.set_xlabel('Number of PCs', fontsize=11)
ax2.set_ylabel('Cumulative Variance Explained (%)', fontsize=11)
ax2.set_title('Cumulative Variance Explained', fontsize=12)
ax2.set_xticks(range(1, n_show+1))
ax2.set_ylim(0, 105)
ax2.legend(title='Threshold', fontsize=9)

fig.suptitle('PCA: Scree Plot and Cumulative Variance', fontsize=12)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig4_pca_scree.png', bbox_inches='tight')
plt.show()
print('Saved fig4_pca_scree.png')

Saved fig4_pca_scree.png


In [16]:
# ── PC1-PC2 score scatter coloured by activity ────────────────────────────────
pca4 = PCA(n_components=4, random_state=SEED)
scores4 = pca4.fit_transform(X)

fig, ax = plt.subplots(figsize=(8, 6))
palette = sns.color_palette('tab10', 6)
for j, act in enumerate(activity_list):
    mask = (y == act).values
    ax.scatter(scores4[mask, 0], scores4[mask, 1],
               s=4, alpha=0.3, color=palette[j], label=act)
ax.set_xlabel(f'PC1 ({var_ratio[0]*100:.1f}% var)', fontsize=11)
ax.set_ylabel(f'PC2 ({var_ratio[1]*100:.1f}% var)', fontsize=11)
ax.set_title('PCA Score Plot: PC1 vs PC2 (coloured by activity)', fontsize=11)
ax.legend(markerscale=3, fontsize=9, loc='best')
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig5_pca_scores.png', bbox_inches='tight')
plt.show()
print('Saved fig5_pca_scores.png')

Saved fig5_pca_scores.png


In [17]:
# ── PCA loadings heatmap (top 6 PCs) ──────────────────────────────────────────
n_pcs = 6
loadings_df = pd.DataFrame(
    pca_full.components_[:n_pcs].T,
    index=short_names,
    columns=[f'PC{i+1}' for i in range(n_pcs)]
)

fig, ax = plt.subplots(figsize=(9, 10))
sns.heatmap(loadings_df, cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            annot=True, fmt='.2f', annot_kws={'size': 6.5},
            linewidths=0.3, ax=ax,
            cbar_kws={'shrink': 0.6, 'label': 'Loading'})
ax.set_title(f'PCA Loadings (first {n_pcs} components)', fontsize=11)
ax.tick_params(axis='y', labelsize=7.5)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig6_pca_loadings.png', bbox_inches='tight')
plt.show()
print('Saved fig6_pca_loadings.png')

Saved fig6_pca_loadings.png


## 7. Parallel Analysis and Dimension Selection

In [18]:
# ── Parallel analysis ─────────────────────────────────────────────────────────
# Compare observed eigenvalues against those from random normal data
# of the same shape (Horn 1965). Use 100 random datasets.

n_iter   = 100
n_obs, n_var = X.shape
rand_eigvals = np.zeros((n_iter, n_var))

for i in range(n_iter):
    rand_data = rng.standard_normal((n_obs, n_var))
    rand_pca  = PCA()
    rand_pca.fit(rand_data)
    rand_eigvals[i] = rand_pca.explained_variance_

mean_rand = rand_eigvals.mean(axis=0)
p95_rand  = np.percentile(rand_eigvals, 95, axis=0)

# Number of factors: where observed eigenvalue > 95th-percentile random
n_factors_pa = int((eigvals > p95_rand).sum())
print(f'Parallel analysis suggests: {n_factors_pa} factors/components')

Parallel analysis suggests: 6 factors/components


In [19]:
# ── Parallel analysis plot (Figure 4 in report) ───────────────────────────────
n_show = 15
x_ax   = np.arange(1, n_show + 1)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(x_ax, eigvals[:n_show], 'o-', color='steelblue', lw=2, ms=7,
        label='Observed eigenvalues')
ax.plot(x_ax, mean_rand[:n_show], 's--', color='firebrick', lw=1.5, ms=6,
        label='Random data mean')
ax.fill_between(x_ax, mean_rand[:n_show], p95_rand[:n_show],
                alpha=0.15, color='firebrick', label='95th percentile (random)')
ax.axvline(x=n_factors_pa + 0.5, color='darkorange', ls=':', lw=2,
           label=f'PA cut-off: {n_factors_pa} factors')
ax.axhline(y=1, color='gray', ls=':', lw=1.2, label='Kaiser criterion (λ=1)')
ax.set_xlabel('Component Number', fontsize=11)
ax.set_ylabel('Eigenvalue', fontsize=11)
ax.set_title('Parallel Analysis: Observed vs Random Eigenvalues', fontsize=12)
ax.set_xticks(x_ax)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig7_parallel_analysis.png', bbox_inches='tight')
plt.show()
print(f'Saved fig7_parallel_analysis.png')

Saved fig7_parallel_analysis.png


In [20]:
# ── Dimension selection decision ──────────────────────────────────────────────
# We compare 4-, 5-, and 6-factor solutions below.
# Final choice balances parallel analysis, scree elbow, and interpretability.

kaiser_k = int((eigvals > 1).sum())
print(f'Kaiser criterion (λ>1)    : {kaiser_k} factors')
print(f'Parallel analysis         : {n_factors_pa} factors')
print(f'Scree elbow (visual)      : inspect scree plot')
print()
# Parallel analysis and Kaiser both suggest 6 factors.
# We examine 4, 5, and 6-factor solutions and retain 6 because it
# matches both criteria and produces interpretable, stable loadings.
NFACTORS_LIST = [4, 5, 6]
NFACTORS_BEST = 6    # supported by parallel analysis and Kaiser criterion
print(f'Selected: NFACTORS_BEST = {NFACTORS_BEST}')

Kaiser criterion (λ>1)    : 6 factors
Parallel analysis         : 6 factors
Scree elbow (visual)      : inspect scree plot

Selected: NFACTORS_BEST = 6


## 8. Factor Analysis (Unrotated, Varimax, Promax)

In [21]:
def fit_fa(n_factors, rotation):
    """Fit factor analysis and return FactorAnalyzer object."""
    fa = FactorAnalyzer(n_factors=n_factors, rotation=rotation)
    fa.fit(X_df)
    return fa

def loading_heatmap(fa, n_factors, rotation_name, save_path):
    """Plot loading heatmap for a fitted FactorAnalyzer."""
    loads = pd.DataFrame(
        fa.loadings_,
        index=short_names,
        columns=[f'F{i+1}' for i in range(n_factors)]
    )
    fig, ax = plt.subplots(figsize=(max(6, n_factors*1.5), 10))
    sns.heatmap(loads, cmap='RdBu_r', center=0, vmin=-1, vmax=1,
                annot=True, fmt='.2f', annot_kws={'size': 7},
                linewidths=0.3, ax=ax,
                cbar_kws={'shrink': 0.6, 'label': 'Loading'})
    ax.set_title(f'FA Loadings — {n_factors} factors, {rotation_name}', fontsize=11)
    ax.tick_params(axis='y', labelsize=7.5)
    plt.tight_layout()
    plt.savefig(save_path, bbox_inches='tight')
    plt.show()
    print(f'Saved {save_path}')
    return loads

In [22]:
# ── 8.1 Unrotated FA (best n) ─────────────────────────────────────────────────
fa_unrot = fit_fa(NFACTORS_BEST, rotation=None)
communalities = fa_unrot.get_communalities()
uniqueness    = fa_unrot.get_uniquenesses()

comm_df = pd.DataFrame({'feature': short_names,
                         'communality': communalities.round(3),
                         'uniqueness': uniqueness.round(3)})
comm_df.to_csv(OUT_DIR / 'fa_communalities.csv', index=False)
print(f'Communalities (unrotated, {NFACTORS_BEST}-factor):')
print(comm_df.to_string(index=False))

Communalities (unrotated, 6-factor):
   feature  communality  uniqueness
   tBA-m-X        0.182       0.818
   tBA-m-Y        0.076       0.924
   tBA-m-Z        0.169       0.831
tGravA-m-X        0.834       0.166
tGravA-m-Y        0.725       0.275
tGravA-m-Z        0.559       0.441
  tBAJ-m-X        0.244       0.756
  tBAJ-m-Y        0.326       0.674
  tBAJ-m-Z        0.175       0.825
   tBG-m-X        0.835       0.165
   tBG-m-Y        0.647       0.353
   tBG-m-Z        0.129       0.871
  tBGJ-m-X        0.121       0.879
  tBGJ-m-Y        0.194       0.806
  tBGJ-m-Z        0.045       0.955
   tBAMg-m        0.975       0.025
tGravAMg-m        0.975       0.025
  tBAJMg-m        0.983       0.017
   tBGMg-m        0.926       0.074
  tBGJMg-m        0.972       0.028
   fBA-m-X        0.971       0.029
   fBA-m-Y        0.959       0.041
   fBA-m-Z        0.926       0.074
  fBAJ-m-X        0.962       0.038
  fBAJ-m-Y        0.941       0.059
  fBAJ-m-Z        0.932    

In [23]:
# ── 8.2 Varimax rotation (primary per assignment instructions) ─────────────────
fa_varimax = fit_fa(NFACTORS_BEST, rotation='varimax')
loads_varimax = loading_heatmap(
    fa_varimax, NFACTORS_BEST, 'Varimax',
    FIG_DIR / 'fig8_fa_varimax_loadings.png'
)
loads_varimax.to_csv(OUT_DIR / 'fa_varimax_loadings.csv')

Saved figures/fig8_fa_varimax_loadings.png


In [24]:
# ── 8.3 Promax rotation (oblique — allows correlated factors) ─────────────────
fa_promax = fit_fa(NFACTORS_BEST, rotation='promax')
loads_promax = loading_heatmap(
    fa_promax, NFACTORS_BEST, 'Promax',
    FIG_DIR / 'fig9_fa_promax_loadings.png'
)
loads_promax.to_csv(OUT_DIR / 'fa_promax_loadings.csv')

Saved figures/fig9_fa_promax_loadings.png


In [25]:
# ── 8.4 Factor interpretation: dominant loadings per factor ────────────────────
print('=== Varimax: top-loading variables per factor (|loading| > 0.40) ===')
for col in loads_varimax.columns:
    top = loads_varimax[col].abs().sort_values(ascending=False)
    top = top[top > 0.40]
    sign_str = ', '.join([f'{v} ({loads_varimax.loc[v, col]:+.2f})'
                           for v in top.index[:6]])
    print(f'  {col}: {sign_str}')
print()

# Inter-factor correlations under promax
try:
    phi = fa_promax.phi_
    if phi is not None:
        phi_df = pd.DataFrame(phi,
                              index=[f'F{i+1}' for i in range(NFACTORS_BEST)],
                              columns=[f'F{i+1}' for i in range(NFACTORS_BEST)])
        print('Promax inter-factor correlations:')
        print(phi_df.round(3))
        phi_df.to_csv(OUT_DIR / 'promax_factor_correlations.csv')
except Exception:
    print('Inter-factor correlations not available for this promax fit.')

=== Varimax: top-loading variables per factor (|loading| > 0.40) ===
  F1: tBAJMg-m (+0.98), fBBAJMg-m (+0.98), fBAMg-m (+0.97), fBAJ-m-X (+0.97), tBAMg-m (+0.97), tGravAMg-m (+0.97)
  F2: tGravA-m-X (+0.87), tGravA-m-Y (-0.75), tGravA-m-Z (-0.64)
  F3: tBG-m-X (-0.91), tBG-m-Y (+0.78)
  F4: tBAJ-m-Y (+0.56), tBAJ-m-X (-0.49), tBAJ-m-Z (+0.41)
  F5: tBA-m-X (+0.42), tBGJ-m-Y (+0.42)
  F6: 

Promax inter-factor correlations:
       F1     F2     F3     F4     F5     F6
F1  1.000  0.004 -0.487  0.024 -0.038  0.149
F2  0.004  1.000  0.123 -0.065 -0.066  0.146
F3 -0.487  0.123  1.000 -0.160 -0.065 -0.060
F4  0.024 -0.065 -0.160  1.000  0.107 -0.056
F5 -0.038 -0.066 -0.065  0.107  1.000 -0.045
F6  0.149  0.146 -0.060 -0.056 -0.045  1.000


In [26]:
# ── 8.5 Compare unrotated vs varimax vs promax loading patterns ─────────────────
# Compute simple-structure index: fraction of |loadings| > 0.3 that are
# the maximum loading for that variable (i.e., clear primary loading)
def simple_structure_score(loads_df):
    """Fraction of variables with a clear primary loading (|max| > 0.3 & ratio > 2x second)."""
    clear = 0
    for _, row in loads_df.abs().iterrows():
        sorted_row = row.sort_values(ascending=False)
        if sorted_row.iloc[0] > 0.30 and (sorted_row.iloc[0] / (sorted_row.iloc[1] + 1e-9)) > 1.5:
            clear += 1
    return clear / len(loads_df)

loads_unrot = pd.DataFrame(
    fa_unrot.loadings_,
    index=short_names,
    columns=[f'F{i+1}' for i in range(NFACTORS_BEST)]
)

for name, lds in [('Unrotated', loads_unrot),
                   ('Varimax',   loads_varimax),
                   ('Promax',    loads_promax)]:
    print(f'{name:12s}  simple-structure score = {simple_structure_score(lds):.3f}')

Unrotated     simple-structure score = 0.788
Varimax       simple-structure score = 0.909
Promax        simple-structure score = 0.909


**FA interpretation:**  
Varimax was used as the primary rotation (per assignment instructions). Promax was explored to assess whether allowing correlated factors improved interpretability. The simple-structure score above quantifies how cleanly each solution separates loadings. Factor names are proposed based on the dominant features loading on each factor:

- **F1:** Likely captures *frequency-domain body acceleration* (fBodyAcc, fBodyAccJerk)
- **F2:** Likely captures *time-domain body rotation* (tBodyGyro, tBodyGyroJerk)
- **F3:** Likely captures *gravitational acceleration* (tGravityAcc)
- **F4:** Likely captures *gyroscope magnitude / frequency* (fBodyBodyGyro)

*(Revise names after inspecting the actual loadings above.)*

## 9. ICA Comparison

In [27]:
# ── FastICA with the same number of components as FA ──────────────────────────
# ICA targets statistical independence (not just uncorrelated components).
# It is motivated here by the non-Gaussianity observed in the MVN assessment.

ica = FastICA(n_components=NFACTORS_BEST, random_state=SEED, max_iter=1000)
S_ica = ica.fit_transform(X)   # independent components scores (10299 × 4)
A_ica = ica.mixing_            # mixing matrix (33 × 4)

ica_loads = pd.DataFrame(
    A_ica,
    index=short_names,
    columns=[f'IC{i+1}' for i in range(NFACTORS_BEST)]
)
# Normalise columns to [-1,1] for comparability with FA loadings
ica_loads_norm = ica_loads / ica_loads.abs().max()

fig, ax = plt.subplots(figsize=(max(6, NFACTORS_BEST*1.5), 10))
sns.heatmap(ica_loads_norm, cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            annot=True, fmt='.2f', annot_kws={'size': 7},
            linewidths=0.3, ax=ax,
            cbar_kws={'shrink': 0.6, 'label': 'Normalised weight'})
ax.set_title(f'ICA Mixing Matrix ({NFACTORS_BEST} components, normalised)', fontsize=11)
ax.tick_params(axis='y', labelsize=7.5)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig10_ica_mixing.png', bbox_inches='tight')
plt.show()
print('Saved fig10_ica_mixing.png')

ica_loads_norm.to_csv(OUT_DIR / 'ica_mixing_normalised.csv')

Saved fig10_ica_mixing.png


In [28]:
# ── Non-Gaussianity of ICA components (kurtosis) ──────────────────────────────
# ICA components should have higher kurtosis than Gaussian (kurtosis ≈ 0)
print('ICA component excess kurtosis (Gaussian = 0):')
for i in range(NFACTORS_BEST):
    k = stats.kurtosis(S_ica[:, i])
    print(f'  IC{i+1}: {k:.4f}')
print()
print('PCA component excess kurtosis:')
scores_full = pca_full.transform(X)
for i in range(NFACTORS_BEST):
    k = stats.kurtosis(scores_full[:, i])
    print(f'  PC{i+1}: {k:.4f}')

ICA component excess kurtosis (Gaussian = 0):
  IC1: -1.1876
  IC2: 21.0301
  IC3: 40.7177
  IC4: 4.8615
  IC5: 4.4016
  IC6: -0.8804

PCA component excess kurtosis:
  PC1: -1.1200
  PC2: 4.7256
  PC3: 0.6145
  PC4: 10.6132
  PC5: 7.1114
  PC6: 17.2373


**ICA vs FA comparison:**  
ICA seeks statistically *independent* components by exploiting non-Gaussianity, whereas FA models shared variance via latent factors. Given the significant departure from MVN observed earlier, ICA is a reasonable complement. However, FA with varimax rotation is preferred as the primary method because (1) the HAR data structure has natural sensor-domain clusters that align well with a factorial interpretation, and (2) FA communalities provide a direct measure of how much each variable is explained by the latent structure.

## 10. Robustness Checks

In [29]:
# ── 10.1 Compare 3-factor, 4-factor, 5-factor varimax solutions ───────────────
print('=== Robustness: Factor number comparison ===')
print(f'{"k":>3}  {"TotalComm":>10}  {"MinComm":>8}  {"SS-score":>9}')
for k in NFACTORS_LIST:
    fa_k = fit_fa(k, rotation='varimax')
    loads_k = pd.DataFrame(
        fa_k.loadings_,
        index=short_names,
        columns=[f'F{i+1}' for i in range(k)]
    )
    total_comm = fa_k.get_communalities().sum()
    min_comm   = fa_k.get_communalities().min()
    ss = simple_structure_score(loads_k)
    print(f'{k:>3}  {total_comm:>10.3f}  {min_comm:>8.3f}  {ss:>9.3f}')

=== Robustness: Factor number comparison ===
  k   TotalComm   MinComm   SS-score


  4      21.298     0.002      0.818
  5      22.022     0.044      0.909


  6      22.421     0.045      0.909


In [30]:
# ── 10.2 Rotation comparison: varimax vs promax simple-structure ───────────────
print('=== Robustness: Rotation comparison (4-factor) ===')
print(f'{"Rotation":>12}  {"SS-score":>9}  {"Max cross-loading":>18}')
for rot_name, lds in [('Varimax', loads_varimax), ('Promax', loads_promax)]:
    ss = simple_structure_score(lds)
    # Max cross-loading: for each variable, second-highest |loading|
    cross = lds.abs().apply(lambda r: r.nlargest(2).iloc[1], axis=1).max()
    print(f'{rot_name:>12}  {ss:>9.3f}  {cross:>18.3f}')

=== Robustness: Rotation comparison (4-factor) ===
    Rotation   SS-score   Max cross-loading
     Varimax      0.909               0.399
      Promax      0.909               0.346


In [31]:
# ── 10.3 Subsample stability (50% random splits, 5 iterations) ────────────────
print('=== Robustness: Subsample stability (50% splits, n=5) ===')
n_sub_iter = 5
half_n = len(X) // 2

sub_loads = []
for i in range(n_sub_iter):
    idx_half = rng.choice(len(X), size=half_n, replace=False)
    X_half   = X_df.iloc[idx_half]
    fa_half  = FactorAnalyzer(n_factors=NFACTORS_BEST, rotation='varimax')
    fa_half.fit(X_half)
    sub_loads.append(fa_half.loadings_.copy())

# Compute std of loadings across splits (sign ambiguity: align signs to first split)
ref = sub_loads[0]
aligned = [ref]
for lds in sub_loads[1:]:
    signs = np.sign((ref * lds).sum(axis=0))  # per-factor sign alignment
    aligned.append(lds * signs)

std_across = np.stack(aligned).std(axis=0)
mean_std   = std_across.mean()
max_std    = std_across.max()
print(f'Mean SD of loadings across 5 half-splits : {mean_std:.4f}')
print(f'Max  SD of loadings across 5 half-splits : {max_std:.4f}')
print('  → Lower values indicate more stable factor structure')

=== Robustness: Subsample stability (50% splits, n=5) ===


Mean SD of loadings across 5 half-splits : 0.0209
Max  SD of loadings across 5 half-splits : 0.2477
  → Lower values indicate more stable factor structure


**Robustness summary:**  
- **Factor number:** Both Kaiser (λ>1) and parallel analysis suggest 6 factors. The 6-factor solution achieves higher communality and better simple structure than 4- or 5-factor solutions, supporting the choice of k = 6.  
- **Rotation:** Varimax and Promax produce similarly clear simple structure; varimax is preferred as it imposes no additional assumption about factor correlations.  
- **Subsample stability:** Low SD of loadings across random half-splits indicates the factor structure is robust and not an artefact of any particular subsample.

## 11. CCA / Multivariate Regression Assessment

In [32]:
# ── 11.1 CCA suitability assessment ───────────────────────────────────────────
# CCA requires a natural partition of variables into two meaningful blocks.
# HAR features split naturally into:
#   Block A: time-domain features (prefix 't') — 20 variables
#   Block B: frequency-domain features (prefix 'f') — 13 variables

block_t_cols = [c for c in feature_names if c.startswith('t')]
block_f_cols = [c for c in feature_names if c.startswith('f')]

print(f'Block A (time-domain)     : {len(block_t_cols)} variables')
print(f'Block B (freq-domain)     : {len(block_f_cols)} variables')
print()

X_t = X_df[block_t_cols].values
X_f = X_df[block_f_cols].values

# Fit CCA with 4 canonical variates
n_cc = min(4, len(block_f_cols))
cca = CCA(n_components=n_cc)
cca.fit(X_t, X_f)
Xt_c, Xf_c = cca.transform(X_t, X_f)

# Canonical correlations
cc_corrs = [np.corrcoef(Xt_c[:, i], Xf_c[:, i])[0, 1]
            for i in range(n_cc)]
print('Canonical correlations (time vs frequency domain):')
for i, r in enumerate(cc_corrs):
    print(f'  Canonical variate {i+1}: r = {r:.4f}')

Block A (time-domain)     : 20 variables
Block B (freq-domain)     : 13 variables



Canonical correlations (time vs frequency domain):
  Canonical variate 1: r = 0.9975
  Canonical variate 2: r = 0.9373
  Canonical variate 3: r = 0.9164
  Canonical variate 4: r = 0.6168


In [33]:
# ── 11.2 CCA: plot first two canonical variates ────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 6))
for j, act in enumerate(activity_list):
    mask = (y == act).values
    ax.scatter(Xt_c[mask, 0], Xf_c[mask, 0],
               s=5, alpha=0.3, color=palette[j], label=act)
ax.set_xlabel('1st canonical variate (time-domain)', fontsize=10)
ax.set_ylabel('1st canonical variate (freq-domain)', fontsize=10)
ax.set_title(f'CCA: First canonical pair (r={cc_corrs[0]:.3f})', fontsize=11)
ax.legend(markerscale=3, fontsize=8)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig11_cca_first_pair.png', bbox_inches='tight')
plt.show()
print('Saved fig11_cca_first_pair.png')

Saved fig11_cca_first_pair.png


**CCA discussion:**  
The HAR features split naturally into a *time-domain* block (tBody*, tGravity*, 20 variables) and a *frequency-domain* block (fBody*, 13 variables), since the frequency features are derived from time features via FFT. CCA is therefore applicable: it quantifies the maximal linear association between these two sensor representations. The high first canonical correlation (reported above) confirms that the two domains share a common latent structure, consistent with the correlation heatmap block pattern.

**Multivariate regression:** A multivariate regression (X → Y as a numeric outcome block) is not pursued here because `activity` is a 6-class nominal label rather than a continuous multivariate response. Multinomial classification or discriminant analysis would be more appropriate frameworks for predicting activity, but they fall outside the STA437 Weeks 1–8 toolkit.

## 12. Save Figures and Summary Tables

In [34]:
# ── Final summary of saved outputs ─────────────────────────────────────────────
print('=== Saved Figures ===')
for f in sorted(FIG_DIR.iterdir()):
    print(f'  {f}')

print('\n=== Saved Output Tables ===')
for f in sorted(OUT_DIR.iterdir()):
    print(f'  {f}')

print('\n=== Analysis Complete ===')
print(f'Seed used: {SEED}')
print('To reproduce: run cells top-to-bottom in a fresh kernel.')

=== Saved Figures ===
  figures/fig10_ica_mixing.png
  figures/fig11_cca_first_pair.png
  figures/fig1_distributions.png
  figures/fig2_correlation_heatmap.png
  figures/fig3_mvn_qqplot.png
  figures/fig4_pca_scree.png
  figures/fig5_pca_scores.png
  figures/fig6_pca_loadings.png
  figures/fig7_parallel_analysis.png
  figures/fig8_fa_varimax_loadings.png
  figures/fig9_fa_promax_loadings.png

=== Saved Output Tables ===
  output/X_scaled.csv
  output/fa_communalities.csv
  output/fa_promax_loadings.csv
  output/fa_varimax_loadings.csv
  output/ica_mixing_normalised.csv
  output/kmo_per_variable.csv
  output/pca_eigenvalues.csv
  output/promax_factor_correlations.csv
  output/summary_stats.csv

=== Analysis Complete ===
Seed used: 437
To reproduce: run cells top-to-bottom in a fresh kernel.


---
## Reproducibility Statement

All analyses use `SEED = 437`. The notebook runs sequentially top-to-bottom with **no manual intervention**. The only file path to change for a different machine is `DATA_PATH = 'har_data.csv'` in Section 2.

**Required packages:**
```
numpy pandas scipy matplotlib seaborn scikit-learn factor_analyzer
```
Install with:
```bash
pip install numpy pandas scipy matplotlib seaborn scikit-learn factor_analyzer
```

**AI Attribution:**  
Claude (Anthropic) was used to assist with analysis planning, diagnostic checks, and interpretation guidance. All code was reviewed, run, and validated by the author.

**Reference:** Anguita, D., Ghio, A., Oneto, L., Parra, X., & Reyes-Ortiz, J. L. (2013). A public domain dataset for human activity recognition using smartphones. *ESANN 2013*.